In [0]:
"""
Analytics Engine
================

Tracks performance metrics for the banking agent.

Metrics tracked:
1. Latency (response time)
2. Success rate (% of requests that succeed)
3. Accuracy (fraud detection accuracy)
4. Cost (cost per request)
5. Safety (transactions blocked, suspicious activity)
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import uuid
import time

# ============================================
# 1. SETUP METRICS TABLES
# ============================================

print("="*60)
print("ANALYTICS ENGINE: Performance Tracking")
print("="*60)

print("\nStep 1: Setting up analytics infrastructure...")

spark.sql("USE banking_agent_db")

# Create metrics table
spark.sql("""
CREATE TABLE IF NOT EXISTS metrics (
    metric_id STRING,
    metric_name STRING,
    metric_value DOUBLE,
    timestamp TIMESTAMP,
    dimension_1 STRING,
    dimension_2 STRING
)
USING DELTA
""")

# Create request logs table
spark.sql("""
CREATE TABLE IF NOT EXISTS request_logs (
    request_id STRING,
    user_id STRING,
    action STRING,
    latency_ms DOUBLE,
    status STRING,
    timestamp TIMESTAMP
)
USING DELTA
""")

# Create cost tracking table
spark.sql("""
CREATE TABLE IF NOT EXISTS cost_tracking (
    cost_id STRING,
    user_id STRING,
    action STRING,
    cost DOUBLE,
    timestamp TIMESTAMP
)
USING DELTA
""")

print("✓ Metrics tables created (metrics, request_logs, cost_tracking)")

# ============================================
# 2. ANALYTICS CLASS
# ============================================

class Analytics:
    """
    Tracks performance metrics for the banking agent.
    
    Metrics tracked:
    - Latency: How fast is the agent? (milliseconds)
    - Success rate: How often does it work? (%)
    - Accuracy: How correct is fraud detection? (%)
    - Cost: How much does each request cost? ($)
    - Safety: How many suspicious transactions? (count)
    
    Example:
        analytics = Analytics()
        
        # Track a request
        analytics.start_timer()
        # ... process request ...
        analytics.end_timer(user_id="user_123", action="transfer")
        
        # Track fraud detection
        analytics.log_accuracy("fraud", actual=1, predicted=1)
        
        # Get summary
        summary = analytics.get_summary()
        print(summary)
    """
    
    def __init__(self):
        """Initialize analytics engine"""
        self.request_start_time = None
    
    # ========================================
    # LATENCY TRACKING
    # ========================================
    
    def start_timer(self) -> None:
        """
        Start timing a request.
        
        Call this at the START of processing.
        """
        self.request_start_time = time.time()
    
    def end_timer(self, user_id: str, action: str, status: str = "success") -> None:
        """
        End timing a request and log latency.
        
        What it does:
        1. Calculate elapsed time (end - start)
        2. Convert to milliseconds
        3. Log to request_logs table
        
        Args:
            user_id: User identifier
            action: What action was performed
            status: "success", "failed", "blocked", etc.
        """
        if self.request_start_time is None:
            return
        
        elapsed_ms = (time.time() - self.request_start_time) * 1000
        request_id = f"req_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        try:
            from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
            
            schema = StructType([
                StructField("request_id", StringType()),
                StructField("user_id", StringType()),
                StructField("action", StringType()),
                StructField("latency_ms", DoubleType()),
                StructField("status", StringType()),
                StructField("timestamp", TimestampType())
            ])
            
            log_data = spark.createDataFrame([
                (request_id, user_id, action, float(elapsed_ms), status, now)
            ], schema=schema)
            
            log_data.write.mode("append").saveAsTable("request_logs")
            
            # Print summary
            print(f"  ⏱️ Request: {action} by {user_id} - {elapsed_ms:.0f}ms")
            
        except Exception as e:
            print(f"  Note: Latency logging: {e}")
        
        self.request_start_time = None
    
    # ========================================
    # ACCURACY TRACKING
    # ========================================
    
    def log_accuracy(self, model_name: str, actual: int, predicted: int) -> None:
        """
        Log prediction accuracy (for ML models).
        
        What it does:
        1. Check if prediction matches actual
        2. Record as metric
        
        Args:
            model_name: "fraud_detector", "sentiment", etc.
            actual: True label (0 or 1)
            predicted: Model's prediction (0 or 1)
            
        Example:
            # Fraud detector: actual fraud, correctly predicted
            analytics.log_accuracy("fraud", actual=1, predicted=1)
            
            # Fraud detector: no fraud, incorrectly predicted fraud
            analytics.log_accuracy("fraud", actual=0, predicted=1)
        """
        metric_id = f"metric_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        # Was prediction correct?
        is_correct = 1.0 if actual == predicted else 0.0
        
        try:
            from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
            
            schema = StructType([
                StructField("metric_id", StringType()),
                StructField("metric_name", StringType()),
                StructField("metric_value", DoubleType()),
                StructField("timestamp", TimestampType()),
                StructField("dimension_1", StringType()),
                StructField("dimension_2", StringType())
            ])
            
            metric_data = spark.createDataFrame([
                (metric_id, "accuracy", is_correct, now, model_name, "")
            ], schema=schema)
            
            metric_data.write.mode("append").saveAsTable("metrics")
            
            result = "✓ CORRECT" if is_correct else "❌ WRONG"
            print(f"  {result}: {model_name} prediction logged")
            
        except Exception as e:
            print(f"  Note: Accuracy logging: {e}")
    
    # ========================================
    # COST TRACKING
    # ========================================
    
    def log_cost(self, user_id: str, action: str, cost: float) -> None:
        """
        Log cost of a request.
        
        What it does:
        - Tracks cost per request
        - Helps understand overall system cost
        - Useful for billing and optimization
        
        Args:
            user_id: User identifier
            action: What action was performed
            cost: Cost in dollars
            
        Example:
            # Request used:
            # - 1 embedding: $0.001
            # - 1 LLM call: $0.01
            # Total: $0.011
            analytics.log_cost("user_123", "transfer", cost=0.011)
        """
        cost_id = f"cost_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        try:
            from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
            
            schema = StructType([
                StructField("cost_id", StringType()),
                StructField("user_id", StringType()),
                StructField("action", StringType()),
                StructField("cost", DoubleType()),
                StructField("timestamp", TimestampType())
            ])
            
            cost_data = spark.createDataFrame([
                (cost_id, user_id, action, float(cost), now)
            ], schema=schema)
            
            cost_data.write.mode("append").saveAsTable("cost_tracking")
            
            print(f"  💰 Cost: ${cost:.4f} for {action}")
            
        except Exception as e:
            print(f"  Note: Cost logging: {e}")
    
    # ========================================
    # SUMMARY STATISTICS
    # ========================================
    
    def get_summary(self) -> Dict:
        """
        Get summary statistics of all metrics.
        
        Returns:
            {
                'avg_latency_ms': 245.5,
                'success_rate': 0.98,
                'accuracy': 0.95,
                'total_cost': 1.23,
                'requests_count': 100,
                'accuracy_count': 95
            }
        """
        try:
            # Get latency stats
            latency_stats = spark.sql("""
            SELECT 
                AVG(latency_ms) as avg_latency,
                MIN(latency_ms) as min_latency,
                MAX(latency_ms) as max_latency,
                COUNT(*) as request_count
            FROM request_logs
            """).collect()
            
            if latency_stats:
                latency_row = latency_stats[0]
                avg_latency = latency_row['avg_latency']
                request_count = latency_row['request_count']
            else:
                avg_latency = 0
                request_count = 0
            
            # Get success rate
            success_stats = spark.sql("""
            SELECT 
                SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) as success_count,
                COUNT(*) as total_count
            FROM request_logs
            """).collect()
            
            if success_stats and request_count > 0:
                success_row = success_stats[0]
                success_rate = success_row['success_count'] / success_row['total_count'] if success_row['total_count'] > 0 else 0
            else:
                success_rate = 0
            
            # Get accuracy
            accuracy_stats = spark.sql("""
            SELECT 
                AVG(metric_value) as accuracy,
                COUNT(*) as accuracy_count
            FROM metrics
            WHERE metric_name = 'accuracy'
            """).collect()
            
            if accuracy_stats:
                accuracy_row = accuracy_stats[0]
                accuracy = accuracy_row['accuracy'] or 0
                accuracy_count = accuracy_row['accuracy_count']
            else:
                accuracy = 0
                accuracy_count = 0
            
            # Get cost
            cost_stats = spark.sql("""
            SELECT 
                SUM(cost) as total_cost,
                AVG(cost) as avg_cost,
                COUNT(*) as cost_count
            FROM cost_tracking
            """).collect()
            
            if cost_stats:
                cost_row = cost_stats[0]
                total_cost = cost_row['total_cost'] or 0.0
                avg_cost = cost_row['avg_cost'] or 0.0
            else:
                total_cost = 0.0
                avg_cost = 0.0
            
            return {
                'avg_latency_ms': float(avg_latency) if avg_latency else 0,
                'request_count': int(request_count),
                'success_rate': float(success_rate),
                'accuracy': float(accuracy),
                'accuracy_count': int(accuracy_count),
                'total_cost': float(total_cost),
                'avg_cost_per_request': float(avg_cost),
            }
            
        except Exception as e:
            print(f"Note: Summary stats error: {e}")
            return {}

# ============================================
# 3. TEST ANALYTICS
# ============================================

print("\n" + "="*60)
print("TESTING ANALYTICS ENGINE")
print("="*60)

analytics = Analytics()

# Test 1: Latency Tracking
print("\nTest 1: Latency Tracking")
print("-"*60)

for i in range(3):
    analytics.start_timer()
    time.sleep(0.1 + i * 0.05)  # Simulate varying response times
    analytics.end_timer(user_id=f"user_{i+1}", action="balance_check", status="success")

# Test 2: Accuracy Tracking
print("\nTest 2: Accuracy Tracking (Fraud Detection)")
print("-"*60)

test_cases = [
    (0, 0, "Correct: Normal transaction classified as normal"),
    (0, 1, "False positive: Normal classified as fraud"),
    (1, 1, "Correct: Fraud classified as fraud"),
    (1, 0, "False negative: Fraud classified as normal"),
    (0, 0, "Correct: Normal transaction classified as normal"),
]

for actual, predicted, description in test_cases:
    print(f"\n{description}")
    analytics.log_accuracy("fraud_detector", actual=actual, predicted=predicted)

# Test 3: Cost Tracking
print("\n" + "-"*60)
print("Test 3: Cost Tracking")
print("-"*60)

test_costs = [
    ("user_1", "balance_check", 0.008),
    ("user_2", "transfer", 0.015),
    ("user_3", "account_settings", 0.006),
]

for user_id, action, cost in test_costs:
    analytics.log_cost(user_id, action, cost)

# Test 4: Get Summary
print("\n" + "-"*60)
print("Test 4: Get Performance Summary")
print("-"*60)

summary = analytics.get_summary()

if summary:
    print(f"\n✓ PERFORMANCE SUMMARY:")
    print(f"  Requests: {summary['request_count']}")
    print(f"  Avg Latency: {summary['avg_latency_ms']:.0f}ms")
    print(f"  Success Rate: {summary['success_rate']:.1%}")
    print(f"  Accuracy: {summary['accuracy']:.1%} ({summary['accuracy_count']} predictions)")
    print(f"  Total Cost: ${summary['total_cost']:.2f}")
    print(f"  Avg Cost/Request: ${summary['avg_cost_per_request']:.4f}")

print("\n" + "="*60)
print("✓ ANALYTICS ENGINE READY")
print("="*60)

# Export
analytics_engine = analytics

print(f"\n✓ Exported for Phase 3:")
print(f"  - analytics_engine (Analytics instance)")
print(f"  - metrics table (Databricks)")
print(f"  - request_logs table (Databricks)")
print(f"  - cost_tracking table (Databricks)")